# Distributed Training

DDP-style gradient sync + ZeRO-1 via `DistributedWrapper`, plus FSDP
(ZeRO-3). `launch` short-circuits for single-process runs so you can smoke
test this cell.

:material-alert-decagram: Gradient sync needs the C backend + NCCL.

In [ ]:
import numpy as np
from SneppX_ALG import (
    Transformer, DistributedWrapper, AdamW, CrossEntropyLoss,
    DistributedSampler, get_world_size, get_rank,
    init_process_group, destroy_process_group,
    FullyShardedDataParallel, FSDPConfig, ShardingStrategy, MixedPrecision,
    launch, Tensor, TensorDataset,
)
from SneppX_ALG.interface_bindings.data_loader import DataLoader

## The training function (one per rank)

In [ ]:
def main():
    init_process_group(backend='nccl')
    rank, world = get_rank(), get_world_size()
    print(f'rank {rank}/{world}')

    model = Transformer(vocab_size=800, dim=256, num_heads=4, num_layers=4,
                        ffn_dim=1024, max_seq_len=64)
    dp_model = DistributedWrapper(model, device='cuda' if world > 1 else 'cpu')

    X = Tensor.randn((256, 64))
    y = Tensor(np.random.randint(0, 800, (256, 64)))
    ds = TensorDataset(X, y)
    sampler = DistributedSampler(ds, num_replicas=world, rank=rank, shuffle=True)
    loader = DataLoader(ds, batch_size=32, sampler=sampler)

    opt = AdamW(dp_model.parameters(), lr=2e-4, weight_decay=0.01)
    for epoch in range(2):
        sampler.set_epoch(epoch)
        for xb, yb in loader:
            logits = dp_model(xb)
            loss = CrossEntropyLoss()(logits.reshape((-1, 800)), yb.reshape((-1,)))
            if get_rank() == 0:
                print('epoch', epoch, 'loss', round(loss.item(), 4))
            opt.zero_grad(); loss.backward()
            dp_model.sync_gradients(); opt.step()
    destroy_process_group()

# Smoke test (single process): world_size collapses to 1
launch(main, num_nodes=1, num_gpus=1) if __name__ == '__main__' else None

## FSDP (ZeRO-3) for larger models

In [ ]:
fsdp_cfg = FSDPConfig(
    sharding=ShardingStrategy.FULL_SHARD,
    mixed_precision=MixedPrecision(param='bf16', reduce='fp32'),
)
base = Transformer(vocab_size=800, dim=512, num_heads=8, num_layers=12,
                   ffn_dim=2048, max_seq_len=128)
sharded = FullyShardedDataParallel(base, fsdp_cfg)
print('FSDP model wrapped:', sharded is not None)